# Agentic-commerce signals in Spotify and Shop Pay / Shopify billing descriptions

**Goal.** Check whether Spotify or Shop Pay / Shopify card descriptions carry agentic-commerce markers around the Muse launch (2026-09-08), and mine descriptor patterns that surged after it.

**Steps**
1. Scope: card sources with descriptions (dedup set from `card_linkagnt_coverage.ipynb`), rows matching `spotify|shopify|shop pay|shoppay|shp*`.
2. Agentic keyword scan, 2026-08-01 to 2026-10-02.
3. Pattern mining: normalise descriptions, compare share before vs after launch, rank by lift.
4. Daily chart of the top surging patterns.
5. Cross-check: Link agentic checkouts (`LINKAGNT*` / `LINK*`) at Shopify stores and Spotify.

## Status as of 2026-10-07 (run via SRS MCP Snowflake; local JWT login broken)

**Scope (Yodlee F3/F4/F6 card, 2026-08-01 to 10-02):** Spotify 319,356 txns / 157,633 members (~5,000/day, flat); Shopify / Shop Pay 20,399 txns / 7,968 members.

**2. Agentic keyword scan: no genuine hits in Spotify or Shop Pay descriptors.**

| Term | Spotify | Shopify / Shop Pay | What the hits are |
|---|---:|---:|---|
| `agnt` | 0 | 0 | |
| `agent` | 0 | 1 | `Card Locked: VGS ** Shopify Agent.. St. Louis` ($0.00, 09-19, F6 card): a card-vault token for a Shopify agent, declined; the only such row in Jun–Oct |
| `gpt` | 3 | 0 | `GPT` inside Spotify `REF #` ids (false positive) |
| `claude` / `openai` / `perplexity` | 0 | 0 | |
| `muse` | 0 | 26 | `SP SHOPIFYMUSEUMCITYNY` (Museum of the City of NY store) |
| `ai` (token) | 0 | 0 | |

**3. Pattern mining. The big post-launch "surges" are billing-descriptor re-labels, not new activity:**
- **Spotify, from 2026-09-10:** `Spotify P46… New York NY` → `SPOTIFY*P46… SPOTIFY.COM NY`.
  - `SPOTIFY.COM NY` went from 0 to 16,283 txns (13.2% of post-launch share).
  - `SPOTIFY NY` share fell from 24.8% to 7.4%.
  - Daily Spotify total is flat (~5,000) and the median amount is unchanged (~$14.2).
- **Shopify, from 2026-09-17:** `SHOPIFY* … ELK GROVE VIL IL` / `SHOPIFY.COM IL` → `SHOPIFY* … NEW YORK NY` / `SHOPIFY.COM NY`.
  - The IL forms went to 0 from the week of 09-21; the NY forms are ~10% shares each.
  - Weekly `SHOPIFY*` volume is flat (~1,700). `SHOPIFY*` is Shopify billing its merchants, not consumer Shop Pay purchases.
- Shop Pay-named consumer descriptors (`SP SHOPPAY…`, `SHOPPAYA`, `SP SHOP PAY PRODUCTION`) are flat (~100/week; lift 0.8–1.9 on small counts).
- Caveat: consumer Shop Pay purchases usually bill as `SP <STORE>` with no Shop Pay marker, so the `shopify|shop pay|shp*` scope only sees stores with "shop" in the name.

**5. Where the agentic signal actually is.** Agentic Link checkouts at Shopify stores (`LINKAGNT* SP <store>`, `LINK* SP <store>`):

| Week | All agentic (Yodlee cards) | At Shopify stores | Spotify |
|---|---:|---:|---:|
| 08-03 | 4 | 0 | 0 |
| 08-10 | 12 | 0 | 0 |
| 08-24 | 16 | 4 | 0 |
| 08-31 | 12 | 0 | 0 |
| 09-07 | 68 | 7 | 0 |
| 09-14 | 82 | 21 | 0 |
| 09-21 | 165 | 21 | 0 |
| 09-28 (lagged) | 128 | 17 | 0 |

**Conclusions for the app**
- Spotify and Shop Pay descriptors contain no agent / AI markers. Don't build on keyword matching there.
- Update descriptor normalisation for the two re-labels (Spotify 09-10, Shopify 09-17); otherwise they look like surges or collapses.
- To track agentic purchases at Shopify merchants, use the Link agentic descriptor followed by `SP ` (13–25% of agentic volume since mid-Sept).

In [ ]:
import os
import pandas as pd
import snowflake.connector
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# Uses a named connection from ~/.snowflake/connections.toml (same pattern as srs-evals).
conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION", "default"),
)

def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

LAUNCH = "2026-09-08"
SEPT = pd.date_range("2026-08-01", "2026-09-30")  # Aug baseline + Sept
START, END = "2026-08-01", "2026-10-02"  # data after 10-02 is incomplete (reporting lag)

## 1. Scope

Card sources with free-text descriptions, deduplicated as in notebook 1: Yodlee feeds 3/4/6 (card + bank) and 279 `PRIME_TAU`. 279 `PRIME` uses `merchant` tags (`spotify`, `shopify`) as well, but it is a subset of Yodlee.

In [ ]:
def src_yodlee(feed, kind):
    return dict(name=f"Yodlee F{feed} {kind.lower()}",
                table=f"SHARE_CE_SPEND_201_PRIME_EX_TAU.FEED{feed}_PANELS.FEED{feed}_{kind}_PANELS_V1_0",
                desc_col="description", date_col="transaction_date", amt_col="amount",
                txn_id_col=f"unique_{kind.lower()}_transaction_id", merchant_col=None)

SOURCES = [
    *[src_yodlee(f, k) for f in (3, 4, 6) for k in ("CARD", "BANK")],
    dict(name="279 PRIME_TAU", table="SHARE_EARNEST_279.PRIME_TAU.SPEND_279_TRANSACTIONS_V3_0",
         desc_col="billing_description", date_col="transaction_date", amt_col="transaction_amount",
         txn_id_col="earnest_transaction_id", merchant_col="merchant"),
]

SCOPE_RE = r"(spotify|shopify|shop ?pay|shp\\*)"

def scope_where(src):
    w = f"REGEXP_LIKE({src['desc_col']}, '.*{SCOPE_RE}.*', 'i')"
    if src["merchant_col"]:
        w = f"({w} OR {src['merchant_col']} IN ('spotify', 'shopify', 'shop_pay'))"
    return w

def family_sql(col):
    return f"IFF({col} ILIKE '%spotify%', 'Spotify', 'Shopify / Shop Pay')"

def scan_range(src, where_sql, start, end):
    return q(f"""
        SELECT '{src['name']}' AS source, {src['txn_id_col']} AS txn_id, {src['date_col']}::DATE AS day,
               {src['amt_col']} AS amount, {src['desc_col']} AS description, {family_sql(src['desc_col'])} AS family
        FROM {src['table']}
        WHERE {src['date_col']} BETWEEN '{start}' AND '{end}' AND {where_sql}
    """)

def scan_daily(src, where_sql, days):
    """One query per day; use for the 279 tables where multi-day ILIKE scans time out."""
    return pd.concat([scan_range(src, where_sql, d.date(), d.date()) for d in days], ignore_index=True)

rows = []
for s in SOURCES:
    w = scope_where(s)
    rows.append(scan_daily(s, w, pd.date_range(START, END)) if s["name"].startswith("279") else scan_range(s, w, START, END))
tx = pd.concat(rows, ignore_index=True).drop_duplicates(["source", "txn_id"])
tx["day"] = pd.to_datetime(tx.day)
tx["period"] = (tx.day >= LAUNCH).map({False: "pre", True: "post"})
tx.groupby(["family", "source"]).agg(n=("txn_id", "size"), first=("day", "min"), last=("day", "max"))

## 2. Agentic keyword scan

Terms: `agnt`, `agent`, `gpt`, `claude`, `muse`, `openai`, `perplexity`, plus `ai` as a whole token only (`(^|[^a-z])ai([^a-z]|$)`), which avoids "paid" and "main".

In [ ]:
TERMS = {t: t for t in ("agnt", "agent", "gpt", "claude", "muse", "openai", "perplexity")}
TERMS["ai (token)"] = r"(^|[^a-z])ai([^a-z]|$)"

low = tx.description.str.lower()
hits = pd.DataFrame({name: low.str.contains(pat, regex=True) for name, pat in TERMS.items()})
display(hits.groupby(tx.family).sum().T)

kw = tx[hits.any(axis=1)].assign(terms=hits[hits.any(axis=1)].apply(lambda r: ", ".join(r.index[r]), axis=1))
kw.groupby(["family", "terms", "description"]).agg(n=("txn_id", "size"), first=("day", "min"), last=("day", "max")) \
  .sort_values("n", ascending=False).head(40)

## 3. Pattern mining (always runs)

Normalise: uppercase, drop trailing `REF #`, `Date`, `Card`, `Merchant Category Code` blocks, drop any token containing a digit or `X` mask, strip punctuation except `*` and `.`, collapse whitespace. Then compare each pattern's share of family volume before vs after launch and rank by lift, with a minimum-volume floor.

In [ ]:
def normalise(s: pd.Series) -> pd.Series:
    return (s.str.upper()
             .str.replace(r"(REF #|DATE |CARD |MERCHANT CATEGORY CODE).*$", "", regex=True)
             .str.replace(r"[^ ]*[0-9X#][^ ]*", " ", regex=True)
             .str.replace(r"[^A-Z*. ]", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())

tx["pattern"] = normalise(tx.description)

MIN_VOLUME = 30

def lift_table(fam: str, top: int = 50) -> pd.DataFrame:
    d = tx[tx.family == fam]
    c = d.groupby(["pattern", "period"]).size().unstack("period").reindex(columns=["pre", "post"]).fillna(0)
    c["pre_share"] = c.pre / c.pre.sum()
    c["post_share"] = c.post / c.post.sum()
    c["lift"] = c.post_share / c.pre_share.replace(0, pd.NA)
    c["new"] = c.pre == 0
    return c[(c.pre + c.post) >= MIN_VOLUME].sort_values("post", ascending=False).head(top)

spotify_lift = lift_table("Spotify")
shop_lift = lift_table("Shopify / Shop Pay")
display(spotify_lift.sort_values(["new", "lift"], ascending=False).head(20))
shop_lift.sort_values(["new", "lift"], ascending=False).head(20)

In [ ]:
# Re-label check: a surging pattern with flat family volume and unchanged price is a descriptor change, not new activity
for fam in tx.family.unique():
    d = tx[tx.family == fam]
    daily = d.groupby("day").agg(total=("txn_id", "size"), median_amount=("amount", "median"))
    print(fam, "| mean daily txns pre/post:",
          round(daily[daily.index < LAUNCH].total.mean()), "/", round(daily[daily.index >= LAUNCH].total.mean()),
          "| median amount pre/post:",
          round(d[d.period == "pre"].amount.median(), 2), "/", round(d[d.period == "post"].amount.median(), 2))

## 4. Daily counts for the top surging patterns

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8))
for ax, (fam, lt) in zip(axes, [("Spotify", spotify_lift), ("Shopify / Shop Pay", shop_lift)]):
    surging = lt[(lt.new | (lt.lift > 1.5)) & (lt.post >= MIN_VOLUME)].index[:6]
    fading = lt[lt.lift < 0.5].index[:2]  # the descriptor each new pattern replaced
    keep = list(surging) + list(fading)
    d = tx[(tx.family == fam) & tx.pattern.isin(keep)]
    d.pivot_table(index="day", columns="pattern", values="txn_id", aggfunc="count").fillna(0).plot(ax=ax, title=f"{fam}: surging / fading descriptor patterns")
    ax.axvline(pd.Timestamp(LAUNCH), color="grey", ls="--", lw=1)
    ax.legend(fontsize=7)
plt.tight_layout()

## 5. Cross-check: Link agentic checkouts at Shopify stores and Spotify

The agentic marker sits in the Link descriptor, not in the store's (`LINKAGNT* SP <store>`, `LINK* SP <store>`, with the `SP ` Shopify prefix). Same agentic definition as notebook 1.

In [ ]:
def agentic_sql(col: str) -> str:
    return (f"({col} ILIKE '%linkagnt*%' OR (REGEXP_LIKE({col}, '(^|.*[^a-z.])link\\\\* .*', 'i') "
            f"AND ({col} ILIKE '%www.link.co%' OR {col} ILIKE '%south san fra%')))")

ag = pd.concat([
    q(f"""
        SELECT '{s['name']}' AS source, {s['date_col']}::DATE AS day, {s['desc_col']} AS description, {s['amt_col']} AS amount
        FROM {s['table']}
        WHERE {s['date_col']} BETWEEN '{START}' AND '{END}' AND {s['desc_col']} ILIKE '%link%' AND {agentic_sql(s['desc_col'])}
    """)
    for s in SOURCES if s["name"].startswith("Yodlee")
], ignore_index=True)
ag["day"] = pd.to_datetime(ag.day)
low = ag.description.str.lower()
ag["at_shopify_store"] = low.str.contains(r"(linkagnt|link)\* ?sp ", regex=True) | low.str.contains("shopify|shop pay", regex=True)
ag["spotify"] = low.str.contains("spotify")
weekly = ag.groupby(pd.Grouper(key="day", freq="W-MON", label="left", closed="left")).agg(
    agentic=("description", "size"), at_shopify_store=("at_shopify_store", "sum"), spotify=("spotify", "sum"))
weekly["shopify_share"] = weekly.at_shopify_store / weekly.agentic
display(weekly)
ag[ag.at_shopify_store].description.str.upper().str.replace(r"\s+", " ", regex=True).value_counts().head(20)